# Week 3 Day 1 – KNN Graph Construction

## Objective

Convert the real-estate dataset into a spatial K-Nearest Neighbor (KNN) graph.

Each property will be represented as a graph node, while edges will connect geographically nearby properties based on Haversine distance.

The resulting graph will provide the structural foundation for spatial embeddings and Graph Neural Network modeling.

In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.neighbors import NearestNeighbors

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
os.makedirs("../data/processed/week-3", exist_ok=True)

print("Week 3 output directory is ready.")

Week 3 output directory is ready.


In [3]:
spatial_df = pd.read_csv("../data/processed/spatial_features.csv")

print("Dataset shape:", spatial_df.shape)
print("\nColumns:")
print(spatial_df.columns.tolist())

Dataset shape: (21613, 30)

Columns:
['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15', 'Sale Year', 'House Age', 'geometry', 'Nearest Neighbor Distance (km)', 'Average Neighbor Distance (km)', 'Neighborhood Average Price', 'Neighborhood Price Difference', 'Nearby Houses (2 km)', 'Price Density']


In [4]:
print(spatial_df[["lat", "long"]].head())

print("\nMissing latitude values:", spatial_df["lat"].isna().sum())
print("Missing longitude values:", spatial_df["long"].isna().sum())

       lat     long
0  47.5112 -122.257
1  47.7210 -122.319
2  47.7379 -122.233
3  47.5208 -122.393
4  47.6168 -122.045

Missing latitude values: 0
Missing longitude values: 0


In [5]:
graph_df = spatial_df.copy()

graph_df = graph_df.reset_index(drop=True)

graph_df["node_id"] = np.arange(len(graph_df))

print("Number of graph nodes:", len(graph_df))

Number of graph nodes: 21613


In [6]:
K = 5

print("Number of nearest neighbors:", K)

Number of nearest neighbors: 5


In [7]:
coordinates = np.radians(graph_df[["lat", "long"]].values)

print("Coordinate matrix shape:", coordinates.shape)

Coordinate matrix shape: (21613, 2)


In [8]:
knn = NearestNeighbors(
    n_neighbors=K + 1,
    metric="haversine",
    algorithm="ball_tree"
)

knn.fit(coordinates)

distances, indices = knn.kneighbors(coordinates)

print("Distance matrix shape:", distances.shape)
print("Neighbor index matrix shape:", indices.shape)

knn = NearestNeighbors(n_neighbors=K + 1,metric="haversine",algorithm="ball_tree")

knn.fit(coordinates)

distances, indices = knn.kneighbors(coordinates)

print("Distance matrix shape:", distances.shape)
print("Neighbor index matrix shape:", indices.shape)

Distance matrix shape: (21613, 6)
Neighbor index matrix shape: (21613, 6)
Distance matrix shape: (21613, 6)
Neighbor index matrix shape: (21613, 6)


In [9]:
EARTH_RADIUS_KM = 6371.0

distances_km = distances * EARTH_RADIUS_KM

print(distances_km[:3])

[[0.         0.04447797 0.05559746 0.07592491 0.08218086 0.16436133]
 [0.         0.07562742 0.08190649 0.13401594 0.13401594 0.15297167]
 [0.         0.14336372 0.1445534  0.14997495 0.16860338 0.19321019]]


In [10]:
edges = []

for source_node in range(len(graph_df)):

    neighbors_added = 0

    for neighbor_position in range(len(indices[source_node])):

        target_node = indices[source_node, neighbor_position]

        # Skip the property itself
        if target_node == source_node:
            continue

        distance_km = distances_km[source_node, neighbor_position]

        edges.append({
            "source": source_node,
            "target": target_node,
            "distance_km": distance_km
        })

        neighbors_added += 1

        # Stop after exactly K valid neighbors
        if neighbors_added == K:
            break

edges_df = pd.DataFrame(edges)

print("Number of directed edges:", len(edges_df))
display(edges_df.head(10))

Number of directed edges: 108065


,source,target,distance_km
0,0,6320,0.044478
1,0,16895,0.055597
2,0,5809,0.075925
3,0,2382,0.082181
4,0,5407,0.164361
5,1,16459,0.075627
6,1,13018,0.081906
7,1,2099,0.134016
8,1,19422,0.134016
9,1,342,0.152972


In [11]:
print("Graph Nodes:", len(graph_df))
print("Graph Edges:", len(edges_df))
print("K:", K)

print("\nExpected edges:", len(graph_df) * K)
print("Actual edges:", len(edges_df))

Graph Nodes: 21613
Graph Edges: 108065
K: 5

Expected edges: 108065
Actual edges: 108065


In [12]:
print(edges_df["distance_km"].describe())

count    108065.000000
mean          0.196325
std           0.300451
min           0.000000
25%           0.088956
50%           0.153740
75%           0.229664
max          23.517976
Name: distance_km, dtype: float64


In [13]:
print("Missing source nodes:", edges_df["source"].isna().sum())
print("Missing target nodes:", edges_df["target"].isna().sum())
print("Missing distances:", edges_df["distance_km"].isna().sum())
print("Negative distances:", (edges_df["distance_km"] < 0).sum())

Missing source nodes: 0
Missing target nodes: 0
Missing distances: 0
Negative distances: 0


In [14]:
node_columns = [
    "node_id",
    "lat",
    "long",
]

available_node_columns = [
    col for col in node_columns
    if col in graph_df.columns
]

node_features_df = graph_df[available_node_columns].copy()

print(node_features_df.head())

   node_id      lat     long
0        0  47.5112 -122.257
1        1  47.7210 -122.319
2        2  47.7379 -122.233
3        3  47.5208 -122.393
4        4  47.6168 -122.045


In [15]:
candidate_features = [
    "node_id",
    "id",
    "bedrooms",
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "floors",
    "waterfront",
    "view",
    "condition",
    "grade",
    "sqft_above",
    "sqft_basement",
    "House Age",
    "yr_built",
    "yr_renovated",
    "lat",
    "long"
]

available_features = [
    col for col in candidate_features
    if col in graph_df.columns
]

node_features_df = graph_df[available_features].copy()

print("Node feature columns:")
print(available_features)

print("\nNode feature shape:", node_features_df.shape)

Node feature columns:
['node_id', 'id', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'House Age', 'yr_built', 'yr_renovated', 'lat', 'long']

Node feature shape: (21613, 18)


In [16]:
graph_targets_df = graph_df[["node_id", "price"]].copy()

print("Graph target shape:", graph_targets_df.shape)
display(graph_targets_df.head())

Graph target shape: (21613, 2)


,node_id,price
0,0,221900.0
1,1,538000.0
2,2,180000.0
3,3,604000.0
4,4,510000.0


In [17]:
edges_df.to_csv("../data/processed/week-3/knn_graph_edges.csv",index=False)

print("KNN graph edges saved successfully.")

KNN graph edges saved successfully.


In [18]:
node_features_df.to_csv("../data/processed/week-3/graph_node_features.csv",index=False)

print("Graph node features saved successfully.")

Graph node features saved successfully.


In [19]:
graph_targets_df.to_csv("../data/processed/week-3/graph_targets.csv",index=False)

print("Graph targets saved successfully.")

Graph targets saved successfully.


In [20]:
saved_edges = pd.read_csv("../data/processed/week-3/knn_graph_edges.csv")

saved_nodes = pd.read_csv("../data/processed/week-3/graph_node_features.csv")

print("Saved edge dataset shape:", saved_edges.shape)
print("Saved node dataset shape:", saved_nodes.shape)

Saved edge dataset shape: (108065, 3)
Saved node dataset shape: (21613, 18)


In [21]:
print("=" * 60)
print("KNN GRAPH SUMMARY")
print("=" * 60)

print(f"Number of Nodes        : {len(saved_nodes)}")
print(f"Number of Edges        : {len(saved_edges)}")
print(f"K Neighbors            : {K}")
print(f"Distance Metric        : Haversine")
print(f"Distance Unit          : Kilometers")

print("=" * 60)

KNN GRAPH SUMMARY
Number of Nodes        : 21613
Number of Edges        : 108065
K Neighbors            : 5
Distance Metric        : Haversine
Distance Unit          : Kilometers


In [22]:
# Validate node IDs
assert saved_edges["source"].between(
    0, len(saved_nodes) - 1
).all(), "Invalid source node detected."

assert saved_edges["target"].between(
    0, len(saved_nodes) - 1
).all(), "Invalid target node detected."

# Validate no self-loops
self_loops = saved_edges[
    saved_edges["source"] == saved_edges["target"]
]

print("Self-loops detected:", len(self_loops))

assert len(self_loops) == 0, \
    "Self-loops detected in the graph."

# Validate distances
assert saved_edges["distance_km"].notna().all(), \
    "Missing distance detected."

assert (saved_edges["distance_km"] >= 0).all(), \
    "Negative distance detected."

# Validate total edge count
assert len(saved_edges) == len(saved_nodes) * K, \
    "Unexpected number of graph edges."

print("✅ KNN graph validation passed successfully.")

Self-loops detected: 0
✅ KNN graph validation passed successfully.
